# Spatio-Temporal ConvLSTM for Cellular Traffic Prediction: Telecom Italia Milano (v2)

Self-contained notebook: the project code is written out by the cells below, so the **only input
to attach is the raw data** (Kaggle dataset `dkgmgo/telecom-italia-milan`). Turn on a **GPU**
accelerator (Settings -> Accelerator -> GPU T4) before running.

Changes from the first run:

- **Standard (mean/std) scaling** instead of min-max, which had squeezed almost all values near 0
  (training loss about 1e-6).
- **Log transform** (`log1p` before scaling): errors become relative, so the busier
  training period and the quieter validation period are on one footing.
- **Calendar inputs:** sin/cos of the time of day and the day of the week.
- **Seasonal baselines:** same time one day earlier and one week earlier, plus skill scores.
- **Relative early-stopping threshold**, so tiny improvements no longer keep a converged run going.
- Val loss reported relative to persistence, `--resume` support, per-sample loss averaging,
  seeded shuffling, and a readable 3-day time-series plot.

Pipeline: write the code -> find the raw files -> preprocess -> train -> evaluate -> compare with run 1.

In [ ]:
import glob, os, shutil, subprocess, sys, json
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
for d in ("/kaggle/working", "/kaggle/temp"):
    os.makedirs(d, exist_ok=True)
    print(f"{d}: {shutil.disk_usage(d).free / 1e9:.1f} GB free")
assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings -> Accelerator) before running."

PROJECT = "/kaggle/working/convlstm_milano"
for sub in ("src", "configs"):
    os.makedirs(f"{PROJECT}/{sub}", exist_ok=True)

## 1. Project code

In [ ]:
%%writefile /kaggle/working/convlstm_milano/src/__init__.py
"""Spatio-temporal ConvLSTM forecasting for the Telecom Italia Milano dataset."""

In [ ]:
%%writefile /kaggle/working/convlstm_milano/src/utils.py
"""Shared utilities: configuration loading, seeding, logging and device selection."""
from __future__ import annotations

import argparse
import json
import logging
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
import yaml

PROJECT_ROOT: Path = Path(__file__).resolve().parents[1]


def _deep_update(base: dict[str, Any], update: dict[str, Any]) -> dict[str, Any]:
    """Recursively merge ``update`` into ``base`` (in place) and return ``base``."""
    for key, value in update.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            _deep_update(base[key], value)
        else:
            base[key] = value
    return base


def resolve_path(path: str | Path) -> Path:
    """Resolve a config path: absolute paths are kept, relative ones are anchored at the project root."""
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_config(path: str | Path, overrides: list[str] | None = None) -> dict[str, Any]:
    """Load a YAML config, following an optional ``_base_`` key and applying CLI overrides.

    Args:
        path: Config file. Relative paths are tried against the CWD first, then the project root.
        overrides: ``"dotted.key=value"`` strings; values are parsed as YAML
            (so ``training.epochs=5`` gives an int and ``data.crop=[0,50,0,50]`` a list).
    """
    path = Path(path)
    if not path.is_absolute() and not path.exists():
        path = PROJECT_ROOT / path
    with open(path, "r", encoding="utf-8") as f:
        cfg: dict[str, Any] = yaml.safe_load(f) or {}

    base = cfg.pop("_base_", None)
    if base is not None:
        cfg = _deep_update(load_config(path.parent / base), cfg)

    for item in overrides or []:
        key, sep, raw = item.partition("=")
        if not sep:
            raise ValueError(f"Override must look like key=value, got {item!r}")
        node = cfg
        *parents, leaf = key.strip().split(".")
        for p in parents:
            node = node.setdefault(p, {})
        node[leaf] = yaml.safe_load(raw)
    return cfg


def add_common_args(parser: argparse.ArgumentParser) -> argparse.ArgumentParser:
    """Add the ``--config`` and ``--set`` arguments shared by every entry point."""
    parser.add_argument("--config", default="configs/default.yaml", help="Path to a YAML config file.")
    parser.add_argument(
        "--set",
        dest="overrides",
        nargs="*",
        default=[],
        metavar="KEY=VALUE",
        help="Override config values, e.g. --set training.epochs=5 data.t_out=3",
    )
    return parser


def set_seed(seed: int) -> None:
    """Seed Python, NumPy and PyTorch RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(preference: str = "auto") -> torch.device:
    """Return CUDA if available (or requested), otherwise CPU."""
    if preference == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(preference)


def get_logger(name: str, log_file: str | Path | None = None, append: bool = False) -> logging.Logger:
    """Create a logger writing to stdout and, optionally, to ``log_file`` (overwritten unless ``append``)."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
    stream = logging.StreamHandler(sys.stdout)
    stream.setFormatter(fmt)
    logger.addHandler(stream)
    if log_file is not None:
        Path(log_file).parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_file, mode="a" if append else "w", encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    return logger


def save_json(obj: Any, path: str | Path) -> None:
    """Write ``obj`` as pretty-printed JSON."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def load_json(path: str | Path) -> Any:
    """Read a JSON file."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
%%writefile /kaggle/working/convlstm_milano/src/dataset.py
"""Preprocessing and PyTorch datasets for the Telecom Italia "Milano" SMS/Call/Internet data.

Pipeline
--------
1. ``preprocess``: stream the daily raw files, aggregate over country codes, and scatter every
   record into a ``(T, C, H, W)`` float32 tensor stored as ``.npy`` (memory-mapped, so the full
   ~1.8 GB, 5-channel tensor never has to fit in RAM at once).
2. ``build_datasets``: chronologically split the time axis, fit the scaling statistics on the
   training segment only, and wrap each segment in a sliding-window ``MilanoTrafficDataset``.

Run ``python -m src.dataset --config configs/default.yaml`` to execute step 1.

Missing values
--------------
The raw data is *sparse*: a (cell, timestamp, country) row is only present when there was
activity, and individual fields are left empty when that activity type was zero. So a missing
value means "no recorded activity", not "unknown". The default ``fill_method: zero`` uses that
interpretation. ``fill_method: ffill_zero`` forward-fills each cell over time and zero-fills
whatever is still missing at the start. Cells with no activity at all in the training period can
also be excluded from the loss and the metrics (``data.mask_inactive``).
"""
from __future__ import annotations

import argparse
import gzip
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

from .utils import add_common_args, get_logger, load_config, load_json, resolve_path, save_json

FEATURES: tuple[str, ...] = ("smsin", "smsout", "callin", "callout", "internet")
LOCAL_TZ = "Europe/Rome"
TENSOR_FILE = "tensor.npy"
META_FILE = "meta.json"

_DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")
# The headerless Harvard Dataverse files have this column order.
_HEADERLESS_COLUMNS = ["cell_id", "time", "countrycode", *FEATURES]
# Header aliases (lower-cased, non-alphanumerics stripped) -> canonical name, for CSV variants
# such as the Kaggle mirror ("datetime,CellID,countrycode,smsin,...").
_COLUMN_ALIASES = {
    "cellid": "cell_id", "squareid": "cell_id", "gridid": "cell_id",
    "datetime": "time", "timeinterval": "time", "time": "time",
    "countrycode": "countrycode",
    "smsin": "smsin", "smsout": "smsout", "callin": "callin", "callout": "callout",
    "internet": "internet",
}


# --------------------------------------------------------------------------------------------
# Raw file handling
# --------------------------------------------------------------------------------------------
def cell_id_to_rowcol(cell_ids: np.ndarray, grid_size: int = 100) -> tuple[np.ndarray, np.ndarray]:
    """Map 1-based Milano ``CellID`` values to ``(row, col)`` grid indices.

    IDs run west->east along a row, then row by row from south to north, so
    ``row = (id - 1) // grid_size`` (row 0 is the southern edge) and ``col = (id - 1) % grid_size``.
    Plot with ``origin="lower"`` to get north at the top.
    """
    idx = np.asarray(cell_ids, dtype=np.int64) - 1
    return idx // grid_size, idx % grid_size


def discover_raw_files(raw_dir: str | Path) -> list[tuple[pd.Timestamp, Path]]:
    """Return ``(date, path)`` for every daily ``sms-call-internet-mi-YYYY-MM-DD.{txt,csv}[.gz]`` file, sorted by date."""
    raw_dir = Path(raw_dir)
    found: list[tuple[pd.Timestamp, Path]] = []
    for path in raw_dir.iterdir():
        name = path.name.lower()
        if not name.endswith((".txt", ".csv", ".txt.gz", ".csv.gz")) or "sms-call-internet" not in name:
            continue
        match = _DATE_RE.search(path.name)
        if match:
            found.append((pd.Timestamp(match.group(1)), path))
    if not found:
        raise FileNotFoundError(f"No 'sms-call-internet-mi-YYYY-MM-DD' files found in {raw_dir}")
    return sorted(found, key=lambda item: item[0])


def _to_local_naive(time_col: pd.Series) -> pd.Series:
    """Parse raw timestamps (epoch milliseconds in UTC, or datetime strings) into naive Milan local time."""
    if pd.api.types.is_numeric_dtype(time_col):
        ts = pd.to_datetime(time_col, unit="ms", utc=True)
    else:
        ts = pd.to_datetime(time_col, utc=False)
    if getattr(ts.dt, "tz", None) is not None:
        ts = ts.dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
    return ts


def read_raw_file(path: str | Path, features: Sequence[str] = FEATURES) -> pd.DataFrame:
    """Read one daily file and aggregate it over country codes.

    Handles both the original headerless, tab-separated Dataverse ``.txt`` files and CSVs with a
    header row (for example the Kaggle mirror).

    Returns:
        DataFrame with columns ``time`` (naive local datetime), ``cell_id`` and one column per
        feature. Each (time, cell) pair appears once; a feature is NaN when it had no records.
    """
    path = Path(path)
    opener = gzip.open if path.suffix.lower() == ".gz" else open
    with opener(path, "rt", encoding="utf-8", errors="replace") as f:
        first_line = f.readline()
    sep = "\t" if "\t" in first_line else ","
    has_header = any(ch.isalpha() for ch in first_line.split(sep)[0])

    if has_header:
        df = pd.read_csv(path, sep=sep)
        df.columns = [_COLUMN_ALIASES.get(re.sub(r"[^a-z0-9]", "", c.lower()), c) for c in df.columns]
    else:
        df = pd.read_csv(path, sep=sep, header=None, names=_HEADERLESS_COLUMNS)

    missing = {"cell_id", "time", *features} - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}; found {list(df.columns)}")

    df["time"] = _to_local_naive(df["time"])
    # min_count=1 keeps NaN when every country row for that field was empty.
    return df.groupby(["time", "cell_id"], as_index=False, sort=False)[list(features)].sum(min_count=1)


def _forward_fill_time(tensor: np.ndarray, rows_per_chunk: int = 10) -> None:
    """Forward-fill NaNs along the time axis in place, per (channel, cell), then zero-fill the rest."""
    T, C, H, W = tensor.shape
    t_idx = np.arange(T)[:, None]
    for c in range(C):
        for r0 in range(0, H, rows_per_chunk):
            r1 = min(r0 + rows_per_chunk, H)
            block = np.array(tensor[:, c, r0:r1, :]).reshape(T, -1)
            last_valid = np.where(np.isnan(block), 0, t_idx)
            np.maximum.accumulate(last_valid, axis=0, out=last_valid)
            block = block[last_valid, np.arange(block.shape[1])]
            tensor[:, c, r0:r1, :] = np.nan_to_num(block, nan=0.0).reshape(T, r1 - r0, W)


def preprocess(
    raw_dir: str | Path,
    processed_dir: str | Path,
    features: Sequence[str] = FEATURES,
    interval_minutes: int = 10,
    grid_size: int = 100,
    max_days: int | None = None,
    fill_method: str = "zero",
    logger: Any | None = None,
) -> Path:
    """Convert the daily raw files into one ``(T, C, H, W)`` float32 tensor plus a metadata file.

    Time index 0 is local midnight of the first file's date. Each record is placed at its absolute
    time index, so rows that spill into a neighbouring day (because of the UTC->local shift) still
    land in the right slot.

    Args:
        raw_dir: Directory containing the daily raw files.
        processed_dir: Output directory for ``tensor.npy`` and ``meta.json``.
        features: Channels to store, in this order.
        interval_minutes: Temporal resolution of the data (10 in the original release).
        grid_size: Grid side length (100 for Milano).
        max_days: Only use the first N days (for quick experiments).
        fill_method: ``"zero"`` or ``"ffill_zero"`` (see the module docstring).

    Returns:
        Path to the written tensor.
    """
    if fill_method not in {"zero", "ffill_zero"}:
        raise ValueError(f"Unknown fill_method {fill_method!r}")
    log = logger.info if logger else print
    features = list(features)
    files = discover_raw_files(raw_dir)
    if max_days:
        files = files[:max_days]

    origin = files[0][0]
    end = files[-1][0] + pd.Timedelta(days=1)
    step = pd.Timedelta(minutes=interval_minutes)
    T = int((end - origin) / step)
    C, H, W = len(features), grid_size, grid_size

    processed_dir = Path(processed_dir)
    processed_dir.mkdir(parents=True, exist_ok=True)
    out_path = processed_dir / TENSOR_FILE
    log(f"Building tensor ({T}, {C}, {H}, {W}) from {len(files)} files "
        f"[{origin.date()} .. {files[-1][0].date()}] -> {out_path}")

    tensor = np.lib.format.open_memmap(out_path, mode="w+", dtype=np.float32, shape=(T, C, H, W))
    tensor[:] = 0.0 if fill_method == "zero" else np.nan

    n_out_of_range = n_bad_cell = n_off_grid = 0
    for i, (date, path) in enumerate(files):
        df = read_raw_file(path, features)
        if i == 0:
            diffs = np.diff(np.sort(df["time"].unique()))
            if len(diffs) and pd.Timedelta(diffs.min()) != step:
                log(f"WARNING: smallest timestamp gap in {path.name} is {pd.Timedelta(diffs.min())}, "
                    f"but interval_minutes={interval_minutes}. Fix data.interval_minutes.")

        offset = (df["time"] - origin) / step
        t = np.floor(offset.to_numpy()).astype(np.int64)
        n_off_grid += int((offset.to_numpy() != t).sum())
        cell = df["cell_id"].to_numpy(np.int64)
        valid_t = (t >= 0) & (t < T)
        valid_cell = (cell >= 1) & (cell <= H * W)
        n_out_of_range += int((~valid_t).sum())
        n_bad_cell += int((valid_t & ~valid_cell).sum())
        keep = valid_t & valid_cell
        t, (r, c) = t[keep], cell_id_to_rowcol(cell[keep], grid_size)

        for ch, feat in enumerate(features):
            vals = df[feat].to_numpy(np.float32)[keep]
            if fill_method == "zero":
                vals = np.nan_to_num(vals, nan=0.0)
            cur = tensor[t, ch, r, c]
            # (time, cell) pairs are unique within a file; accumulate in case a neighbouring file
            # contributed to the same slot.
            tensor[t, ch, r, c] = np.where(np.isnan(cur), vals, cur + np.nan_to_num(vals, nan=0.0))
        log(f"  [{i + 1}/{len(files)}] {path.name}: {keep.sum():,} (time, cell) records")

    if fill_method == "ffill_zero":
        log("Forward-filling missing values along time, then zero-filling the remainder ...")
        _forward_fill_time(tensor)
    tensor.flush()

    if n_out_of_range or n_bad_cell or n_off_grid:
        log(f"Note: dropped {n_out_of_range:,} rows outside the time range, {n_bad_cell:,} rows with "
            f"invalid CellID; {n_off_grid:,} rows were not aligned to the time grid (floored).")

    meta = {
        "features": features,
        "shape": [T, C, H, W],
        "origin": origin.isoformat(),
        "interval_minutes": interval_minutes,
        "grid_size": grid_size,
        "fill_method": fill_method,
        "days": [d.date().isoformat() for d, _ in files],
    }
    save_json(meta, processed_dir / META_FILE)
    log(f"Done. Metadata written to {processed_dir / META_FILE}")
    return out_path


# --------------------------------------------------------------------------------------------
# Normalisation and statistics
# --------------------------------------------------------------------------------------------
SCALERS = ("minmax", "standard")
CALENDAR_CHANNELS = 4  # sin/cos of the time of day, sin/cos of the position in the week


@dataclass
class Normalizer:
    """Per-channel affine scaling ``(x - offset) / scale``, with an optional ``log1p`` applied first.

    * ``kind="minmax"``: ``offset = min``, ``scale = max - min`` (values end up in [0, 1]).
    * ``kind="standard"``: ``offset = mean``, ``scale = std`` (zero mean, unit variance).

    Traffic is heavy-tailed: a handful of city-centre cells are hundreds of times busier than the
    median cell. Min-max scaling therefore squeezes almost every value into the bottom 1% of the
    range and the loss ends up around 1e-6; standardising keeps values and losses near 1.

    ``offset`` and ``scale`` are indexed by channel in the *processed tensor*, so the same object
    serves any subset of input or target channels via the ``channels`` argument.
    """

    offset: np.ndarray
    scale: np.ndarray
    log_transform: bool = False
    kind: str = "minmax"

    def _params(self, channels: Sequence[int], like: np.ndarray | torch.Tensor) -> tuple[Any, Any]:
        lo = self.offset[list(channels)].astype(np.float32)
        scale = self.scale[list(channels)].astype(np.float32)
        scale = np.where(scale > 0, scale, 1.0).astype(np.float32)
        lo, scale = lo.reshape(-1, 1, 1), scale.reshape(-1, 1, 1)
        if isinstance(like, torch.Tensor):
            return (torch.as_tensor(lo, dtype=like.dtype, device=like.device),
                    torch.as_tensor(scale, dtype=like.dtype, device=like.device))
        return lo, scale

    def normalize(self, x: np.ndarray | torch.Tensor, channels: Sequence[int]) -> Any:
        """Scale ``x`` of shape ``(..., len(channels), H, W)``."""
        if self.log_transform:
            x = torch.log1p(x) if isinstance(x, torch.Tensor) else np.log1p(x)
        lo, scale = self._params(channels, x)
        return (x - lo) / scale

    def denormalize(self, x: np.ndarray | torch.Tensor, channels: Sequence[int]) -> Any:
        """Inverse of :meth:`normalize`, returning values in the original units."""
        lo, scale = self._params(channels, x)
        x = x * scale + lo
        if self.log_transform:
            x = torch.expm1(x) if isinstance(x, torch.Tensor) else np.expm1(x)
        return x

    def state_dict(self) -> dict[str, Any]:
        return {"kind": self.kind, "offset": self.offset.tolist(), "scale": self.scale.tolist(),
                "log_transform": self.log_transform}

    @classmethod
    def from_state_dict(cls, state: dict[str, Any]) -> "Normalizer":
        if "mins" in state:  # checkpoints written before the ``standard`` scaler existed
            mins = np.asarray(state["mins"], dtype=np.float64)
            return cls(mins, np.asarray(state["maxs"], dtype=np.float64) - mins, bool(state["log_transform"]), "minmax")
        return cls(np.asarray(state["offset"], dtype=np.float64), np.asarray(state["scale"], dtype=np.float64),
                   bool(state["log_transform"]), state["kind"])


def compute_train_statistics(
    tensor: np.ndarray, t_end: int, target_channels: Sequence[int], log_transform: bool = False,
    scaler: str = "minmax", chunk: int = 256,
) -> tuple[Normalizer, np.ndarray]:
    """Fit the normalizer and the mean activity map on ``tensor[:t_end]`` (the training segment only).

    Returns:
        ``(normalizer, activity)``, where ``activity`` is the ``(H, W)`` mean of the target
        channels over the training period. Cells where it is 0 had no activity at all.
    """
    if scaler not in SCALERS:
        raise ValueError(f"Unknown data.scaler {scaler!r}; choose one of {SCALERS}")
    C, H, W = tensor.shape[1:]
    mins = np.full(C, np.inf)
    maxs = np.full(C, -np.inf)
    total = np.zeros(C)
    total_sq = np.zeros(C)
    activity = np.zeros((H, W), dtype=np.float64)
    for start in range(0, t_end, chunk):
        block = np.asarray(tensor[start:min(start + chunk, t_end)], dtype=np.float64)
        activity += block[:, list(target_channels)].sum(axis=(0, 1))
        if log_transform:
            block = np.log1p(block)
        mins = np.minimum(mins, block.min(axis=(0, 2, 3)))
        maxs = np.maximum(maxs, block.max(axis=(0, 2, 3)))
        total += block.sum(axis=(0, 2, 3))
        total_sq += (block ** 2).sum(axis=(0, 2, 3))
    activity /= t_end * len(target_channels)
    if scaler == "minmax":
        offset, scale = mins, maxs - mins
    else:
        n = t_end * H * W
        offset = total / n
        scale = np.sqrt(np.maximum(total_sq / n - offset ** 2, 0.0))
    return Normalizer(offset, scale, log_transform, scaler), activity.astype(np.float32)


def calendar_features(t_idx: np.ndarray, origin: pd.Timestamp, interval_minutes: int) -> np.ndarray:
    """``(len(t_idx), 4)`` float32: sin/cos of the time of day and of the position in the week.

    One hour of history cannot tell 08:00 (traffic about to rise) from 20:00 (about to fall); these
    features give the model the clock and the weekday.
    """
    minutes = (origin.dayofweek * 1440 + origin.hour * 60 + origin.minute
               + np.asarray(t_idx, dtype=np.int64) * interval_minutes)
    day = 2 * np.pi * (minutes % 1440) / 1440
    week = 2 * np.pi * (minutes % 10080) / 10080
    return np.stack([np.sin(day), np.cos(day), np.sin(week), np.cos(week)], axis=1).astype(np.float32)


def split_bounds(num_steps: int, ratios: Sequence[float]) -> list[tuple[int, int]]:
    """Chronological ``[(start, end), ...]`` time ranges for train/val/test."""
    if len(ratios) != 3 or abs(sum(ratios) - 1.0) > 1e-6:
        raise ValueError(f"split must be three ratios summing to 1, got {ratios}")
    n_train = int(num_steps * ratios[0])
    n_val = int(num_steps * ratios[1])
    return [(0, n_train), (n_train, n_train + n_val), (n_train + n_val, num_steps)]


# --------------------------------------------------------------------------------------------
# Dataset
# --------------------------------------------------------------------------------------------
class MilanoTrafficDataset(Dataset):
    """Sliding windows over one chronological segment of the processed tensor.

    Every window lies entirely inside ``[t_start, t_end)``, so no target from a later split ever
    appears in an earlier one. Item ``i`` is::

        X: (t_in,  C_in,  H, W)   frames t0 .. t0+t_in-1         (normalised; 4 calendar channels
                                  are appended when ``calendar`` is given)
        Y: (t_out, C_out, H, W)   frames t0+t_in .. t0+t_in+t_out-1 (normalised)

    with ``t0 = t_start + i * stride``. The tensor is opened lazily as a memmap inside each worker,
    so DataLoader workers do not pickle the whole array.
    """

    def __init__(
        self,
        tensor_path: str | Path,
        t_start: int,
        t_end: int,
        t_in: int,
        t_out: int,
        input_channels: Sequence[int],
        target_channels: Sequence[int],
        normalizer: Normalizer,
        stride: int = 1,
        crop: Sequence[int] | None = None,
        calendar: tuple[pd.Timestamp, int] | None = None,
    ) -> None:
        self.tensor_path = str(tensor_path)
        self.t_start, self.t_end = t_start, t_end
        self.t_in, self.t_out, self.stride = t_in, t_out, stride
        self.input_channels = list(input_channels)
        self.target_channels = list(target_channels)
        self.normalizer = normalizer
        self.crop = tuple(crop) if crop else None
        self.calendar = calendar  # (origin, interval_minutes), or None for no calendar channels
        self.num_input_channels = len(self.input_channels) + (CALENDAR_CHANNELS if calendar else 0)
        self.num_windows = (t_end - t_start - t_in - t_out) // stride + 1
        if self.num_windows <= 0:
            raise ValueError(f"Segment [{t_start}, {t_end}) is too short for t_in={t_in}, t_out={t_out}")
        self._array: np.ndarray | None = None

    def __len__(self) -> int:
        return self.num_windows

    def _data(self) -> np.ndarray:
        if self._array is None:
            arr = np.load(self.tensor_path, mmap_mode="r")
            if self.crop:
                r0, r1, c0, c1 = self.crop
                arr = arr[:, :, r0:r1, c0:c1]
            self._array = arr
        return self._array

    def target_time_index(self, i: int) -> int:
        """Absolute time index of the first target frame of window ``i``."""
        return self.t_start + i * self.stride + self.t_in

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        arr = self._data()
        t0 = self.t_start + i * self.stride
        x = np.asarray(arr[t0:t0 + self.t_in][:, self.input_channels], dtype=np.float32)
        y = np.asarray(arr[t0 + self.t_in:t0 + self.t_in + self.t_out][:, self.target_channels], dtype=np.float32)
        x = self.normalizer.normalize(x, self.input_channels).astype(np.float32)
        y = self.normalizer.normalize(y, self.target_channels).astype(np.float32)
        if self.calendar is not None:
            feats = calendar_features(np.arange(t0, t0 + self.t_in), *self.calendar)
            planes = np.broadcast_to(feats[:, :, None, None], (self.t_in, CALENDAR_CHANNELS, *x.shape[2:]))
            x = np.concatenate([x, planes], axis=1)
        return torch.from_numpy(x), torch.from_numpy(y)

    def raw_target_frames(self, t_idx: np.ndarray) -> np.ndarray:
        """Target channels at absolute time indices ``t_idx`` (any shape), in original units.

        Used by the seasonal baselines, which look up the frame one day or one week before a target.
        """
        t_idx = np.asarray(t_idx)
        frames = np.asarray(self._data()[t_idx.ravel()][:, self.target_channels], dtype=np.float32)
        return frames.reshape(*t_idx.shape, *frames.shape[1:])


@dataclass
class DataBundle:
    """Everything produced by :func:`build_datasets`."""

    datasets: dict[str, MilanoTrafficDataset]
    normalizer: Normalizer
    mask: torch.Tensor | None  # (H, W) bool over the (cropped) grid, or None
    activity: np.ndarray  # (H, W) mean training activity of the target channels (cropped)
    meta: dict[str, Any]
    input_channels: list[int]
    target_channels: list[int]


def _channel_indices(names: Sequence[str], available: Sequence[str]) -> list[int]:
    unknown = [n for n in names if n not in available]
    if unknown:
        raise ValueError(f"Features {unknown} not in the processed tensor {list(available)}; re-run preprocessing")
    return [list(available).index(n) for n in names]


def build_datasets(cfg: dict[str, Any], normalizer: Normalizer | None = None) -> DataBundle:
    """Create train/val/test datasets from the processed tensor described by ``cfg``.

    Args:
        cfg: Full config dict.
        normalizer: Fitted normalizer to reuse (e.g. from a checkpoint). If ``None``, it is fitted
            on the training segment.
    """
    dcfg = cfg["data"]
    processed_dir = resolve_path(cfg["paths"]["processed_dir"])
    tensor_path = processed_dir / TENSOR_FILE
    if not tensor_path.exists():
        raise FileNotFoundError(f"{tensor_path} not found. Run `python -m src.dataset` first.")
    meta = load_json(processed_dir / META_FILE)
    input_channels = _channel_indices(dcfg["input_features"], meta["features"])
    target_channels = _channel_indices(dcfg["target_features"], meta["features"])

    tensor = np.load(tensor_path, mmap_mode="r")
    bounds = split_bounds(tensor.shape[0], dcfg["split"])
    fitted, activity = compute_train_statistics(
        tensor, bounds[0][1], target_channels, bool(dcfg.get("log_transform", False)),
        normalizer.kind if normalizer else dcfg.get("scaler", "minmax"))
    normalizer = normalizer or fitted

    crop = dcfg.get("crop")
    if crop:
        r0, r1, c0, c1 = crop
        activity = activity[r0:r1, c0:c1]
    mask = torch.from_numpy(activity > 0) if dcfg.get("mask_inactive", True) else None
    calendar = ((pd.Timestamp(meta["origin"]), int(meta["interval_minutes"]))
                if dcfg.get("time_features", False) else None)

    datasets = {
        name: MilanoTrafficDataset(
            tensor_path, start, end, dcfg["t_in"], dcfg["t_out"], input_channels, target_channels,
            normalizer, stride=dcfg.get("stride", 1) if name == "train" else 1, crop=crop, calendar=calendar)
        for name, (start, end) in zip(("train", "val", "test"), bounds)
    }
    return DataBundle(datasets, normalizer, mask, activity, meta, input_channels, target_channels)


def build_dataloaders(cfg: dict[str, Any], datasets: dict[str, MilanoTrafficDataset]) -> dict[str, DataLoader]:
    """Wrap datasets in DataLoaders. Only the training loader is shuffled."""
    workers = int(cfg["data"].get("num_workers", 0))
    # A dedicated generator makes the shuffle order depend only on the seed.
    generator = torch.Generator().manual_seed(int(cfg.get("seed", 42)))
    return {
        name: DataLoader(
            ds,
            batch_size=cfg["training"]["batch_size"],
            shuffle=(name == "train"),
            generator=generator if name == "train" else None,
            num_workers=workers,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=workers > 0,
            drop_last=False,
        )
        for name, ds in datasets.items()
    }


def main() -> None:
    parser = add_common_args(argparse.ArgumentParser(description="Preprocess raw Milano files into a tensor."))
    args = parser.parse_args()
    cfg = load_config(args.config, args.overrides)
    dcfg = cfg["data"]
    preprocess(
        raw_dir=resolve_path(cfg["paths"]["raw_dir"]),
        processed_dir=resolve_path(cfg["paths"]["processed_dir"]),
        features=dcfg["features"],
        interval_minutes=dcfg["interval_minutes"],
        grid_size=dcfg["grid_size"],
        max_days=dcfg.get("max_days"),
        fill_method=dcfg.get("fill_method", "zero"),
        logger=get_logger("preprocess"),
    )


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/convlstm_milano/src/model.py
"""ConvLSTM forecaster: ConvLSTMCell -> stacked ConvLSTM encoder -> convolutional prediction head.

The cell follows Shi et al. (2015), "Convolutional LSTM Network", in the widely used
single-convolution form: input and hidden state are concatenated and one ``Conv2d`` produces all
four gates. Peephole connections are omitted.
"""
from __future__ import annotations

from typing import Any, Sequence

import torch
from torch import nn


class ConvLSTMCell(nn.Module):
    """A single ConvLSTM cell.

    Args:
        input_dim: Channels of the input frame.
        hidden_dim: Channels of the hidden and cell states.
        kernel_size: Odd ``(kh, kw)`` convolution kernel; "same" padding keeps H and W.
        bias: Whether the gate convolution has a bias.
    """

    def __init__(self, input_dim: int, hidden_dim: int, kernel_size: tuple[int, int] = (3, 3), bias: bool = True) -> None:
        super().__init__()
        kernel_size = tuple(kernel_size)
        if any(k % 2 == 0 for k in kernel_size):
            raise ValueError(f"kernel_size must be odd for same padding, got {kernel_size}")
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(
            input_dim + hidden_dim, 4 * hidden_dim, kernel_size,
            padding=(kernel_size[0] // 2, kernel_size[1] // 2), bias=bias)
        if bias:
            # A forget-gate bias of 1 is a standard trick that helps gradients flow early in training.
            with torch.no_grad():
                self.conv.bias.zero_()
                self.conv.bias[hidden_dim:2 * hidden_dim].fill_(1.0)

    def forward(self, input_tensor: torch.Tensor, cur_state: tuple[torch.Tensor, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
        """Advance one time step.

        Args:
            input_tensor: ``(B, input_dim, H, W)``.
            cur_state: ``(h_cur, c_cur)``, each ``(B, hidden_dim, H, W)``.

        Returns:
            ``(h_next, c_next)``.
        """
        h_cur, c_cur = cur_state
        gates = self.conv(torch.cat([input_tensor, h_cur], dim=1))
        cc_i, cc_f, cc_o, cc_g = torch.split(gates, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

    def init_hidden(self, batch_size: int, image_size: tuple[int, int], device: torch.device, dtype: torch.dtype) -> tuple[torch.Tensor, torch.Tensor]:
        """Zero-initialised ``(h, c)``."""
        shape = (batch_size, self.hidden_dim, *image_size)
        return torch.zeros(shape, device=device, dtype=dtype), torch.zeros(shape, device=device, dtype=dtype)


class ConvLSTM(nn.Module):
    """Multi-layer ConvLSTM over a ``(B, T, C, H, W)`` sequence.

    Args:
        input_dim: Input channels ``C``.
        hidden_dim: Hidden channels, either one int for every layer or a list with one per layer.
        kernel_size: Kernel for every layer, or a list with one per layer.
        num_layers: Number of stacked cells.
        bias: Gate-convolution bias.
        return_all_layers: If True, return outputs and states for every layer; otherwise only the last.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int | Sequence[int],
        kernel_size: tuple[int, int] | Sequence[tuple[int, int]] = (3, 3),
        num_layers: int = 1,
        bias: bool = True,
        return_all_layers: bool = False,
    ) -> None:
        super().__init__()
        hidden_dims = list(hidden_dim) if isinstance(hidden_dim, (list, tuple)) else [hidden_dim] * num_layers
        if kernel_size and isinstance(kernel_size[0], (list, tuple)):
            kernel_sizes = [tuple(k) for k in kernel_size]
        else:
            kernel_sizes = [tuple(kernel_size)] * num_layers
        if not len(hidden_dims) == len(kernel_sizes) == num_layers:
            raise ValueError("hidden_dim and kernel_size lists must have num_layers entries")

        self.num_layers = num_layers
        self.hidden_dims = hidden_dims
        self.return_all_layers = return_all_layers
        self.cells = nn.ModuleList(
            ConvLSTMCell(input_dim if i == 0 else hidden_dims[i - 1], hidden_dims[i], kernel_sizes[i], bias)
            for i in range(num_layers))

    def forward(
        self, x: torch.Tensor, hidden_state: list[tuple[torch.Tensor, torch.Tensor]] | None = None
    ) -> tuple[list[torch.Tensor], list[tuple[torch.Tensor, torch.Tensor]]]:
        """Run the sequence through every layer.

        Args:
            x: ``(B, T, C, H, W)``.
            hidden_state: Optional initial ``(h, c)`` per layer; zeros by default.

        Returns:
            ``(layer_outputs, last_states)``. ``layer_outputs[l]`` has shape ``(B, T, hidden_l, H, W)``
            and ``last_states[l] = (h_T, c_T)``. Both contain only the last layer unless
            ``return_all_layers`` is set.
        """
        B, T, _, H, W = x.shape
        if hidden_state is None:
            hidden_state = [cell.init_hidden(B, (H, W), x.device, x.dtype) for cell in self.cells]

        layer_outputs: list[torch.Tensor] = []
        last_states: list[tuple[torch.Tensor, torch.Tensor]] = []
        cur_input = x
        for layer, cell in enumerate(self.cells):
            h, c = hidden_state[layer]
            outputs = []
            for t in range(T):
                h, c = cell(cur_input[:, t], (h, c))
                outputs.append(h)
            cur_input = torch.stack(outputs, dim=1)
            layer_outputs.append(cur_input)
            last_states.append((h, c))

        if not self.return_all_layers:
            return layer_outputs[-1:], last_states[-1:]
        return layer_outputs, last_states


class PredictionHead(nn.Module):
    """Map the final hidden state to ``t_out`` future frames.

    A 1x1 convolution projects ``hidden_dim -> t_out * out_channels`` (direct multi-horizon
    forecasting). An optional stack of 3x3 convolutions then adds a residual correction that
    sharpens the spatial structure of the output.
    """

    def __init__(self, hidden_dim: int, out_channels: int, t_out: int = 1, refine_layers: int = 0, refine_channels: int = 32) -> None:
        super().__init__()
        self.t_out, self.out_channels = t_out, out_channels
        n_out = t_out * out_channels
        self.proj = nn.Conv2d(hidden_dim, n_out, kernel_size=1)
        self.refine: nn.Module | None = None
        if refine_layers > 0:
            layers: list[nn.Module] = []
            ch = n_out + hidden_dim
            for _ in range(refine_layers - 1):
                layers += [nn.Conv2d(ch, refine_channels, 3, padding=1), nn.ReLU(inplace=True)]
                ch = refine_channels
            layers.append(nn.Conv2d(ch, n_out, 3, padding=1))
            self.refine = nn.Sequential(*layers)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """``(B, hidden_dim, H, W) -> (B, t_out, out_channels, H, W)``."""
        out = self.proj(h)
        if self.refine is not None:
            out = out + self.refine(torch.cat([out, h], dim=1))
        B, _, H, W = out.shape
        return out.view(B, self.t_out, self.out_channels, H, W)


class ConvLSTMForecaster(nn.Module):
    """ConvLSTM encoder plus prediction head.

    Args:
        input_dim: Input channels.
        output_dim: Target channels.
        t_out: Number of future frames to predict.
        hidden_dim, kernel_size, num_layers, bias: Passed to :class:`ConvLSTM`.
        refine_layers, refine_channels: Passed to :class:`PredictionHead`.
        residual_channels: Optional positions, within the *input* channels, of each target channel.
            When given, the network predicts the change from the last observed frame, i.e.
            ``y_hat = x[:, -1, residual_channels] + head(h_T)``. This is an optional experiment: in
            early tests with plain min-max scaling it tended to stall at the persistence solution,
            so it is off by default.
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        t_out: int = 1,
        hidden_dim: int | Sequence[int] = 64,
        kernel_size: tuple[int, int] = (3, 3),
        num_layers: int = 2,
        bias: bool = True,
        refine_layers: int = 0,
        refine_channels: int = 32,
        residual_channels: Sequence[int] | None = None,
    ) -> None:
        super().__init__()
        self.encoder = ConvLSTM(input_dim, hidden_dim, kernel_size, num_layers, bias)
        self.head = PredictionHead(self.encoder.hidden_dims[-1], output_dim, t_out, refine_layers, refine_channels)
        if residual_channels is not None and len(residual_channels) != output_dim:
            raise ValueError("residual_channels needs one entry per output channel")
        self.register_buffer(
            "residual_index",
            torch.tensor(list(residual_channels), dtype=torch.long) if residual_channels is not None else None,
            persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """``(B, T_in, C_in, H, W) -> (B, T_out, C_out, H, W)``."""
        _, last_states = self.encoder(x)
        out = self.head(last_states[-1][0])
        if self.residual_index is not None:
            out = out + x[:, -1:, self.residual_index]
        return out


def build_model(cfg: dict[str, Any], input_dim: int, output_dim: int, residual_channels: Sequence[int] | None = None) -> ConvLSTMForecaster:
    """Instantiate :class:`ConvLSTMForecaster` from the ``model`` and ``data`` sections of the config."""
    m = cfg["model"]
    return ConvLSTMForecaster(
        input_dim=input_dim,
        output_dim=output_dim,
        t_out=cfg["data"]["t_out"],
        hidden_dim=m["hidden_dim"],
        kernel_size=tuple(m["kernel_size"]),
        num_layers=m["num_layers"],
        bias=m.get("bias", True),
        refine_layers=m.get("refine_layers", 0),
        refine_channels=m.get("refine_channels", 32),
        residual_channels=residual_channels if m.get("residual", False) else None,
    )


def residual_positions(input_channels: Sequence[int], target_channels: Sequence[int]) -> list[int] | None:
    """Position of each target channel within the input channels, or None if any target is not an input."""
    if not all(t in input_channels for t in target_channels):
        return None
    return [list(input_channels).index(t) for t in target_channels]

In [ ]:
%%writefile /kaggle/working/convlstm_milano/src/train.py
"""Train the ConvLSTM forecaster with early stopping, LR scheduling and best-checkpoint saving.

Usage::

    python -m src.train --config configs/default.yaml [--resume] [--set training.epochs=20 ...]
"""
from __future__ import annotations

import argparse
import contextlib
import csv
import math
import time
from pathlib import Path
from typing import Any, Sequence

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader

from .dataset import build_dataloaders, build_datasets
from .model import build_model, residual_positions
from .utils import add_common_args, get_device, get_logger, load_config, resolve_path, save_json, set_seed

LAST_CHECKPOINT = "last_checkpoint.pt"


def masked_loss(
    pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor | None, kind: str = "mse", huber_delta: float = 1.0
) -> torch.Tensor:
    """MSE or Huber loss averaged over the cells where ``mask`` (an ``(H, W)`` bool) is True."""
    if kind == "mse":
        err = (pred - target) ** 2
    elif kind == "huber":
        err = F.huber_loss(pred, target, reduction="none", delta=huber_delta)
    else:
        raise ValueError(f"Unknown loss {kind!r}")
    if mask is None:
        return err.mean()
    m = mask.to(err.dtype)
    return (err * m).sum() / (m.sum() * (err.numel() // m.numel())).clamp_min(1)


class EarlyStopping:
    """Stop when the monitored value has not improved *noticeably* for ``patience`` epochs.

    Any improvement is a new best (and is checkpointed), but only an improvement of more than
    ``min_delta + min_delta_rel * best`` resets the patience counter. Without the relative
    threshold, improvements in the seventh significant digit keep a converged run alive.
    """

    def __init__(self, patience: int = 10, min_delta: float = 0.0, min_delta_rel: float = 0.0) -> None:
        self.patience, self.min_delta, self.min_delta_rel = patience, min_delta, min_delta_rel
        self.best = math.inf
        self.reference = math.inf  # last value that counted as a noticeable improvement
        self.bad_epochs = 0

    def step(self, value: float) -> bool:
        """Record ``value``; return True if it is a new best."""
        if value < self.reference * (1 - self.min_delta_rel) - self.min_delta:
            self.reference, self.bad_epochs = value, 0
        else:
            self.bad_epochs += 1
        if value < self.best:
            self.best = value
            return True
        return False

    @property
    def should_stop(self) -> bool:
        return self.bad_epochs >= self.patience

    def state_dict(self) -> dict[str, float]:
        return {"best": self.best, "reference": self.reference, "bad_epochs": self.bad_epochs}

    def load_state_dict(self, state: dict[str, float]) -> None:
        self.best, self.reference, self.bad_epochs = state["best"], state["reference"], int(state["bad_epochs"])


def run_epoch(
    model: nn.Module | None,
    loader: DataLoader,
    device: torch.device,
    loss_cfg: dict[str, Any],
    mask: torch.Tensor | None,
    optimizer: torch.optim.Optimizer | None = None,
    scaler: torch.amp.GradScaler | None = None,
    use_amp: bool = False,
    grad_clip: float | None = None,
    max_batches: int | None = None,
    persistence_channels: Sequence[int] | None = None,
) -> float:
    """Run one pass over ``loader``. Trains if ``optimizer`` is given, otherwise only evaluates.

    With ``model=None`` the persistence forecast (the last input frame of ``persistence_channels``,
    repeated for every horizon) is scored instead, which gives the loss a reference point.

    Returns:
        Mean loss per sample.
    """
    training = optimizer is not None
    if model is not None:
        model.train(training)
    total, n = 0.0, 0
    grad_ctx = contextlib.nullcontext() if training else torch.no_grad()
    with grad_ctx:
        for b, (x, y) in enumerate(loader):
            if max_batches is not None and b >= max_batches:
                break
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if model is None:
                pred = x[:, -1:, list(persistence_channels)].expand_as(y)
            else:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    pred = model(x)
            loss = masked_loss(pred.float(), y, mask, loss_cfg["loss"], loss_cfg.get("huber_delta", 1.0))
            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                else:
                    loss.backward()
                if grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
            # Weight by batch size so a short final batch does not count as much as a full one.
            total += loss.item() * x.shape[0]
            n += x.shape[0]
    return total / max(n, 1)


def train(cfg: dict[str, Any], resume: bool = False) -> Path:
    """Train a model according to ``cfg`` and return the path of the best checkpoint.

    With ``resume=True`` and a ``last_checkpoint.pt`` in the output directory, training continues
    from the end of the last finished epoch (model, optimiser, scheduler and early-stopping state).
    """
    out_dir = resolve_path(cfg["paths"]["output_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    last_path = out_dir / LAST_CHECKPOINT
    resume = resume and last_path.exists()
    logger = get_logger("train", out_dir / "train.log", append=resume)
    set_seed(cfg.get("seed", 42))
    device = get_device(cfg.get("device", "auto"))
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True  # input shapes are fixed, so let cuDNN pick the fastest kernels
    tcfg = cfg["training"]
    save_json(cfg, out_dir / "config_used.json")

    bundle = build_datasets(cfg)
    loaders = build_dataloaders(cfg, bundle.datasets)
    mask = bundle.mask.to(device) if bundle.mask is not None else None
    logger.info(f"Device: {device} | windows train/val/test = "
                + "/".join(str(len(bundle.datasets[s])) for s in ("train", "val", "test")))
    if mask is not None:
        logger.info(f"Loss mask: {int(mask.sum())}/{mask.numel()} cells active in the training period")

    residual = residual_positions(bundle.input_channels, bundle.target_channels)
    if cfg["model"].get("residual", False) and residual is None:
        logger.info("model.residual requested, but some target features are not inputs; disabling it")
    model = build_model(cfg, bundle.datasets["train"].num_input_channels, len(bundle.target_channels), residual).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    logger.info(f"Model: {n_params:,} parameters")

    optimizer = torch.optim.Adam(model.parameters(), lr=tcfg["lr"], weight_decay=tcfg.get("weight_decay", 0.0))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=tcfg.get("scheduler_factor", 0.5),
        patience=tcfg.get("scheduler_patience", 5), min_lr=tcfg.get("min_lr", 1e-6))
    use_amp = bool(tcfg.get("amp", True)) and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    stopper = EarlyStopping(tcfg.get("patience", 10), tcfg.get("min_delta", 0.0), tcfg.get("min_delta_rel", 0.0))
    ckpt_path = out_dir / "best_model.pt"
    history_path = out_dir / "history.csv"

    start_epoch = 1
    if resume:
        state = torch.load(last_path, map_location=device, weights_only=True)
        model.load_state_dict(state["model_state"])
        optimizer.load_state_dict(state["optimizer_state"])
        scheduler.load_state_dict(state["scheduler_state"])
        if scaler is not None and state.get("scaler_state"):
            scaler.load_state_dict(state["scaler_state"])
        stopper.load_state_dict(state["stopper_state"])
        start_epoch = state["epoch"] + 1
        logger.info(f"Resumed from {last_path} after epoch {state['epoch']} (best val loss {stopper.best:.4e})")
    else:
        with open(history_path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["epoch", "train_loss", "val_loss", "lr", "seconds",
                                    "train_vs_persistence", "val_vs_persistence"])

    # The loss of "predict no change" on each split. Raw losses from different periods are not
    # comparable (the December validation period is easier than November, so val < train even for
    # persistence); as a fraction of the same split's persistence loss they are, and anything below
    # 1.0 beats persistence.
    persistence_loss = persistence_train = None
    if residual is not None:
        persistence_loss = run_epoch(None, loaders["val"], device, tcfg, mask,
                                     max_batches=tcfg.get("max_eval_batches"), persistence_channels=residual)
        persistence_train = run_epoch(None, loaders["train"], device, tcfg, mask,
                                      max_batches=tcfg.get("max_train_batches"), persistence_channels=residual)
        logger.info(f"Reference: persistence scores a train loss of {persistence_train:.4e} "
                    f"and a val loss of {persistence_loss:.4e}")

    def ratio(loss: float, reference: float | None) -> float | str:
        return round(loss / reference, 5) if reference else ""

    def vs_persistence(val_loss: float) -> str:
        return f" ({val_loss / persistence_loss:.3f}x persistence)" if persistence_loss else ""

    def vs_persistence(val_loss: float) -> str:
        return f" ({val_loss / persistence_loss:.3f}x persistence)" if persistence_loss else ""

    max_seconds = tcfg.get("max_hours") and tcfg["max_hours"] * 3600
    run_start = time.time()
    for epoch in range(start_epoch, tcfg["epochs"] + 1):
        if stopper.should_stop:  # a resumed run that had already stopped
            break
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]  # the rate this epoch trains with
        train_loss = run_epoch(model, loaders["train"], device, tcfg, mask, optimizer, scaler, use_amp,
                               tcfg.get("grad_clip"), tcfg.get("max_train_batches"))
        val_loss = run_epoch(model, loaders["val"], device, tcfg, mask, use_amp=use_amp,
                             max_batches=tcfg.get("max_eval_batches"))
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        improved = stopper.step(val_loss)
        if improved:
            torch.save({
                "model_state": model.state_dict(),
                "config": cfg,
                "epoch": epoch,
                "val_loss": val_loss,
                "normalizer": bundle.normalizer.state_dict(),
                "input_channels": bundle.input_channels,
                "target_channels": bundle.target_channels,
                "residual_channels": residual if cfg["model"].get("residual", False) else None,
                "features": bundle.meta["features"],
            }, ckpt_path)
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict() if scaler is not None else None,
            "stopper_state": stopper.state_dict(),
            "epoch": epoch,
        }, last_path)
        logger.info(f"Epoch {epoch:3d} | train {train_loss:.4e} | val {val_loss:.4e}{vs_persistence(val_loss)} | "
                    f"lr {lr:.2e} | {elapsed:.1f}s{'  * saved best' if improved else ''}")
        with open(history_path, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([epoch, train_loss, val_loss, lr, round(elapsed, 2),
                                    ratio(train_loss, persistence_train), ratio(val_loss, persistence_loss)])

        if stopper.should_stop:
            logger.info(f"Early stopping: no noticeable val improvement for {stopper.patience} epochs")
            break
        # Stop before the next epoch would exceed the time budget (e.g. a Kaggle session limit).
        if max_seconds and time.time() - run_start + elapsed > max_seconds:
            logger.info(f"Stopping: time budget of {tcfg['max_hours']} h reached (continue with --resume)")
            break

    logger.info(f"Best val loss {stopper.best:.4e}{vs_persistence(stopper.best)}; checkpoint: {ckpt_path}")
    return ckpt_path


def main() -> None:
    parser = add_common_args(argparse.ArgumentParser(description="Train the ConvLSTM forecaster."))
    parser.add_argument("--resume", action="store_true",
                        help="Continue from <output_dir>/last_checkpoint.pt if it exists")
    args = parser.parse_args()
    train(load_config(args.config, args.overrides), args.resume)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/convlstm_milano/src/evaluate.py
"""Evaluate a trained checkpoint on the test split against naive baselines, and make plots.

Metrics (RMSE, MAE, MAPE) are computed in the original units after denormalisation, per
prediction horizon and per target channel. Only cells that were active during training count
when ``data.mask_inactive`` is on. MAPE only counts cells whose true value exceeds
``evaluation.mape_threshold``, because it is undefined or explosive near zero.

Baselines:
  * ``Persistence``: repeat the last observed frame for every horizon.
  * ``WindowMean``: the mean of the ``t_in`` input frames (historical average).
  * ``SeasonalDaily`` / ``SeasonalWeekly``: the frame at the same time one day / one week earlier.

The first two need every target feature to also be an input feature; the seasonal ones need the
test period to start at least one day / one week into the data.

Usage::

    python -m src.evaluate --config configs/default.yaml [--checkpoint path/to/best_model.pt]
"""
from __future__ import annotations

import argparse
import csv
from pathlib import Path
from typing import Any, Sequence

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402
from matplotlib.ticker import MaxNLocator  # noqa: E402

from .dataset import Normalizer, build_dataloaders, build_datasets  # noqa: E402
from .model import build_model  # noqa: E402
from .utils import add_common_args, get_device, get_logger, load_config, resolve_path, save_json, set_seed  # noqa: E402

# Plot styling: fixed categorical order, single-hue sequential ramps, recessive axes.
SURFACE = "#fcfcfb"
TEXT, TEXT_MUTED = "#0b0b0b", "#52514e"
SERIES = {"Actual": "#2a78d6", "ConvLSTM": "#eb6834", "Persistence": "#1baf7a"}
SEQ_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
SEQ_ORANGE = LinearSegmentedColormap.from_list("seq_orange", ["#fcfcfb", "#f9d3c0", "#f19a72", "#eb6834", "#b9471c", "#7a2c0f"])


class MetricAccumulator:
    """Streaming RMSE / MAE / MAPE per (horizon, channel), so the test set never has to sit in memory."""

    def __init__(self, t_out: int, n_channels: int, mape_threshold: float) -> None:
        shape = (t_out, n_channels)
        self.sse, self.sae, self.n = np.zeros(shape), np.zeros(shape), np.zeros(shape)
        self.sape, self.n_ape = np.zeros(shape), np.zeros(shape)
        self.mape_threshold = mape_threshold

    def update(self, pred: torch.Tensor, true: torch.Tensor, mask: torch.Tensor | None) -> None:
        """Add a batch. ``pred``/``true``: ``(B, T_out, C, H, W)`` in original units; ``mask``: ``(H, W)`` bool."""
        pred, true = pred.double().flatten(-2), true.double().flatten(-2)
        if mask is not None:
            keep = mask.flatten()
            pred, true = pred[..., keep], true[..., keep]
        err = pred - true
        dims = (0, 3)
        self.sse += (err ** 2).sum(dims).cpu().numpy()
        self.sae += err.abs().sum(dims).cpu().numpy()
        self.n += err.shape[0] * err.shape[3]
        ape_ok = true > self.mape_threshold
        self.sape += torch.where(ape_ok, err.abs() / true.clamp_min(1e-12), torch.zeros_like(err)).sum(dims).cpu().numpy()
        self.n_ape += ape_ok.sum(dims).cpu().numpy()

    @staticmethod
    def _summarise(sse: np.ndarray, sae: np.ndarray, n: np.ndarray, sape: np.ndarray, n_ape: np.ndarray) -> dict[str, float]:
        return {
            "RMSE": float(np.sqrt(sse.sum() / n.sum())),
            "MAE": float(sae.sum() / n.sum()),
            "MAPE_%": float(100 * sape.sum() / n_ape.sum()) if n_ape.sum() else float("nan"),
        }

    def results(self, channel_names: Sequence[str]) -> dict[str, Any]:
        """Overall, per-horizon and per-channel metrics."""
        arrays = (self.sse, self.sae, self.n, self.sape, self.n_ape)
        return {
            "overall": self._summarise(*arrays),
            "per_horizon": [
                {"horizon": h + 1, **self._summarise(*(a[h] for a in arrays))} for h in range(self.sse.shape[0])],
            "per_channel": {
                name: self._summarise(*(a[:, c] for a in arrays)) for c, name in enumerate(channel_names)},
        }


def _style_axes(ax: plt.Axes) -> None:
    ax.set_facecolor(SURFACE)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c9c8c3")
    ax.tick_params(colors=TEXT_MUTED, labelsize=9)
    ax.grid(True, color="#ecebe7", linewidth=0.8)
    ax.set_axisbelow(True)


def plot_heatmaps(samples: list[dict[str, Any]], channel: str, units_label: str, out_path: Path) -> None:
    """One row per sample: actual, ConvLSTM prediction and absolute error (first horizon)."""
    fig, axes = plt.subplots(len(samples), 3, figsize=(13, 4.1 * len(samples)), squeeze=False, facecolor=SURFACE)
    for row, s in enumerate(samples):
        vmax = float(np.percentile(s["actual"], 99.5)) or 1.0
        err = np.abs(s["pred"] - s["actual"])
        panels = [
            (s["actual"], f"Actual · {s['time']}", SEQ_BLUE, vmax),
            (s["pred"], "ConvLSTM prediction", SEQ_BLUE, vmax),
            (err, "Absolute error", SEQ_ORANGE, float(np.percentile(err, 99.5)) or 1.0),
        ]
        for ax, (img, title, cmap, vm) in zip(axes[row], panels):
            im = ax.imshow(img, origin="lower", cmap=cmap, vmin=0, vmax=vm, interpolation="nearest")
            ax.set_title(title, color=TEXT, fontsize=11, loc="left")
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(False)
            cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
            cb.outline.set_visible(False)
            cb.ax.tick_params(colors=TEXT_MUTED, labelsize=8)
            cb.set_label(units_label, color=TEXT_MUTED, fontsize=8)
    fig.suptitle(f"{channel}: predicted vs actual (north up, color capped at the 99.5th percentile)",
                 color=TEXT, fontsize=12, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120, facecolor=SURFACE)
    plt.close(fig)


def plot_cell_series(times: Sequence[pd.Timestamp], series: dict[str, np.ndarray], title: str, units_label: str, out_path: Path) -> None:
    """Time series of actual vs predicted vs persistence for one cell (first horizon)."""
    fig, ax = plt.subplots(figsize=(13, 4.2), facecolor=SURFACE)
    _style_axes(ax)
    # The prediction is drawn last and on top; persistence is a thin reference line underneath.
    for name in ("Persistence", "Actual", "ConvLSTM"):
        if name in series:
            ax.plot(times, series[name], color=SERIES[name], linewidth=1.8 if name != "Persistence" else 1.0,
                    label=name, alpha=1.0 if name != "Persistence" else 0.8)
    ax.set_title(title, color=TEXT, fontsize=12, loc="left")
    ax.set_ylabel(units_label, color=TEXT_MUTED)
    ax.set_ylim(bottom=0)
    leg = ax.legend(frameon=False, loc="upper left", ncol=len(series))
    for text in leg.get_texts():
        text.set_color(TEXT)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(out_path, dpi=120, facecolor=SURFACE)
    plt.close(fig)


def plot_history(history_csv: Path, out_path: Path) -> None:
    """Training and validation loss per epoch."""
    hist = pd.read_csv(history_csv)
    if hist.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 4.2), facecolor=SURFACE)
    _style_axes(ax)
    if "val_vs_persistence" in hist and hist["val_vs_persistence"].notna().any():
        # Each split's loss divided by persistence on the same split: train and validation are
        # then on one scale, and the dashed line at 1.0 is "no better than predicting no change".
        ax.plot(hist["epoch"], hist["train_vs_persistence"], color=SERIES["Actual"], linewidth=2, label="Train")
        ax.plot(hist["epoch"], hist["val_vs_persistence"], color=SERIES["ConvLSTM"], linewidth=2, label="Validation")
        ax.axhline(1.0, color=SERIES["Persistence"], linewidth=1.4, linestyle="--", label="Persistence")
        top = max(1.1, float(np.nanpercentile(hist[["train_vs_persistence", "val_vs_persistence"]].to_numpy(), 95)) * 1.05)
        ax.set_ylim(0, top)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("Epoch", color=TEXT_MUTED)
        ax.set_ylabel("loss ÷ persistence loss", color=TEXT_MUTED)
        ax.set_title("Loss relative to persistence on the same split (lower is better)", color=TEXT, fontsize=12, loc="left")
        leg = ax.legend(frameon=False)
        for text in leg.get_texts():
            text.set_color(TEXT)
        fig.tight_layout()
        fig.savefig(out_path, dpi=120, facecolor=SURFACE)
        plt.close(fig)
        return
    ax.plot(hist["epoch"], hist["train_loss"], color=SERIES["Actual"], linewidth=2, label="Train")
    ax.plot(hist["epoch"], hist["val_loss"], color=SERIES["ConvLSTM"], linewidth=2, label="Validation")
    ax.set_yscale("log")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Epoch", color=TEXT_MUTED)
    ax.set_title("Loss (normalised scale)", color=TEXT, fontsize=12, loc="left")
    leg = ax.legend(frameon=False)
    for text in leg.get_texts():
        text.set_color(TEXT)
    fig.tight_layout()
    fig.savefig(out_path, dpi=120, facecolor=SURFACE)
    plt.close(fig)


def _pick_cell(activity: np.ndarray, mask: torch.Tensor | None, rng: np.random.Generator, fixed: Sequence[int] | None) -> tuple[int, int]:
    """Use the configured cell if given, else a random cell from the 10% most active ones."""
    if fixed:
        return int(fixed[0]), int(fixed[1])
    valid = activity if mask is None else np.where(mask.numpy(), activity, -np.inf)
    threshold = np.percentile(valid[np.isfinite(valid)], 90)
    candidates = np.argwhere(valid >= threshold)
    r, c = candidates[rng.integers(len(candidates))]
    return int(r), int(c)


@torch.no_grad()
def evaluate(cfg: dict[str, Any], checkpoint: str | Path | None = None) -> dict[str, Any]:
    """Evaluate ``checkpoint`` (default ``<output_dir>/best_model.pt``) on the test split."""
    out_dir = resolve_path(cfg["paths"]["output_dir"])
    ckpt_path = resolve_path(checkpoint) if checkpoint else out_dir / "best_model.pt"
    logger = get_logger("evaluate", out_dir / "evaluate.log")
    device = get_device(cfg.get("device", "auto"))
    ecfg = cfg["evaluation"]
    set_seed(cfg.get("seed", 42))

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    # Use the data/model settings the model was trained with; paths and evaluation settings come
    # from the current config.
    run_cfg = {**ckpt["config"], "paths": cfg["paths"], "evaluation": ecfg}
    run_cfg["training"] = {**ckpt["config"]["training"], "batch_size": cfg["training"]["batch_size"]}
    normalizer = Normalizer.from_state_dict(ckpt["normalizer"])

    bundle = build_datasets(run_cfg, normalizer=normalizer)
    test_ds = bundle.datasets["test"]
    loader = build_dataloaders(run_cfg, {"test": test_ds})["test"]
    in_ch, tgt_ch = bundle.input_channels, bundle.target_channels
    target_names = [bundle.meta["features"][i] for i in tgt_ch]
    residual = ckpt.get("residual_channels")

    model = build_model(run_cfg, test_ds.num_input_channels, len(tgt_ch), residual).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    logger.info(f"Loaded {ckpt_path} (epoch {ckpt['epoch']}, val loss {ckpt['val_loss']:.4e}); "
                f"{len(test_ds)} test windows on {device}")

    t_out = run_cfg["data"]["t_out"]
    baseline_pos = [in_ch.index(t) for t in tgt_ch] if all(t in in_ch for t in tgt_ch) else None
    mask = bundle.mask
    mask_dev = mask.to(device) if mask is not None else None
    def new_acc() -> MetricAccumulator:
        return MetricAccumulator(t_out, len(tgt_ch), ecfg.get("mape_threshold", 1.0))

    accs = {"ConvLSTM": new_acc()}
    if baseline_pos is not None:
        accs["Persistence"], accs["WindowMean"] = new_acc(), new_acc()
    else:
        logger.info("Persistence/WindowMean skipped: not every target feature is an input feature")
    # Seasonal baselines: the same time one day / one week earlier. Looking back is legitimate at
    # forecast time, so these frames may come from the training or validation period.
    steps_per_day = round(24 * 60 / bundle.meta["interval_minutes"])
    seasonal = {name: lag for name, lag in (("SeasonalDaily", steps_per_day), ("SeasonalWeekly", 7 * steps_per_day))
                if test_ds.target_time_index(0) - lag >= 0}
    for name in seasonal:
        accs[name] = new_acc()

    rng = np.random.default_rng(cfg.get("seed", 42))
    cell = _pick_cell(bundle.activity, mask, rng, ecfg.get("plot_cell"))
    sample_ids = set(rng.choice(len(test_ds), size=min(ecfg.get("num_plot_samples", 3), len(test_ds)), replace=False).tolist())
    samples: list[dict[str, Any]] = []
    cell_series: dict[str, list[float]] = {"Actual": [], "ConvLSTM": [], "Persistence": []}
    origin = pd.Timestamp(bundle.meta["origin"])
    step = pd.Timedelta(minutes=bundle.meta["interval_minutes"])

    offset = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        pred_raw = normalizer.denormalize(pred.float(), tgt_ch).clamp_min(0)  # traffic is non-negative
        y_raw = normalizer.denormalize(y, tgt_ch)
        accs["ConvLSTM"].update(pred_raw, y_raw, mask_dev)

        persist = None
        if baseline_pos is not None:
            x_raw = normalizer.denormalize(x[:, :, baseline_pos], tgt_ch)
            persist = x_raw[:, -1:].expand(-1, t_out, -1, -1, -1)
            accs["Persistence"].update(persist, y_raw, mask_dev)
            accs["WindowMean"].update(x_raw.mean(dim=1, keepdim=True).expand(-1, t_out, -1, -1, -1), y_raw, mask_dev)

        if seasonal:
            # Window i targets frames t_i .. t_i + t_out - 1, with consecutive t_i (test stride is 1).
            t_idx = (test_ds.target_time_index(offset) + np.arange(x.shape[0])[:, None] + np.arange(t_out)[None, :])
            for name, lag in seasonal.items():
                frames = torch.from_numpy(test_ds.raw_target_frames(t_idx - lag)).to(device)
                accs[name].update(frames, y_raw, mask_dev)

        r, c = cell
        cell_series["Actual"] += y_raw[:, 0, 0, r, c].cpu().tolist()
        cell_series["ConvLSTM"] += pred_raw[:, 0, 0, r, c].cpu().tolist()
        if persist is not None:
            cell_series["Persistence"] += persist[:, 0, 0, r, c].cpu().tolist()

        for b in range(x.shape[0]):
            if offset + b in sample_ids:
                samples.append({
                    "actual": y_raw[b, 0, 0].cpu().numpy(),
                    "pred": pred_raw[b, 0, 0].cpu().numpy(),
                    "time": (origin + step * test_ds.target_time_index(offset + b)).strftime("%Y-%m-%d %H:%M"),
                })
        offset += x.shape[0]

    results = {name: acc.results(target_names) for name, acc in accs.items()}
    # Skill score: the fraction of each baseline's error that the ConvLSTM removes (positive = better).
    ours = results["ConvLSTM"]["overall"]
    results["ConvLSTM"]["skill_vs_baselines_%"] = {
        name: {m: round(100 * (1 - ours[m] / res["overall"][m]), 2) for m in ("RMSE", "MAE")}
        for name, res in results.items() if name != "ConvLSTM"}
    results["notes"] = {
        "units": "original dataset units (unitless activity counts)",
        "mask_inactive": mask is not None,
        "active_cells": int(mask.sum()) if mask is not None else None,
        "mape_threshold": ecfg.get("mape_threshold", 1.0),
        "test_windows": len(test_ds),
        "checkpoint": str(ckpt_path),
    }
    save_json(results, out_dir / "metrics.json")

    rows = []
    for name, res in results.items():
        if name == "notes":
            continue
        for h in res["per_horizon"]:
            rows.append({"model": name, **h})
    with open(out_dir / "metrics_per_horizon.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["model", "horizon", "RMSE", "MAE", "MAPE_%"])
        writer.writeheader()
        writer.writerows(rows)

    logger.info(f"{'model':<14} {'horizon':>7} {'RMSE':>12} {'MAE':>12} {'MAPE %':>9}")
    for row in rows:
        logger.info(f"{row['model']:<14} {row['horizon']:>7} {row['RMSE']:>12.4f} {row['MAE']:>12.4f} {row['MAPE_%']:>9.2f}")
    for name, res in results.items():
        if name != "notes":
            o = res["overall"]
            logger.info(f"{name:<14} overall  RMSE {o['RMSE']:.4f} | MAE {o['MAE']:.4f} | MAPE {o['MAPE_%']:.2f}%")
    for name, skill in results["ConvLSTM"]["skill_vs_baselines_%"].items():
        logger.info(f"ConvLSTM vs {name:<14} RMSE {-skill['RMSE']:+.1f}% | MAE {-skill['MAE']:+.1f}%")

    channel = target_names[0]
    units = f"{channel} activity"
    plot_heatmaps(sorted(samples, key=lambda s: s["time"]), channel, units, out_dir / "heatmaps.png")
    # Only the last `plot_steps` points: ten days of 10-minute data in one panel is an unreadable band.
    n_plot = min(int(ecfg.get("plot_steps") or len(test_ds)), len(test_ds))
    times = [origin + step * test_ds.target_time_index(i) for i in range(len(test_ds) - n_plot, len(test_ds))]
    series = {k: np.asarray(v[-n_plot:]) for k, v in cell_series.items() if v}
    crop = run_cfg["data"].get("crop")
    r_abs, c_abs = (cell[0] + crop[0], cell[1] + crop[2]) if crop else cell
    cell_id = r_abs * bundle.meta["grid_size"] + c_abs + 1
    plot_cell_series(times, series, f"{channel} at CellID {cell_id} (row {r_abs}, col {c_abs}), 1-step-ahead test predictions",
                     units, out_dir / "cell_timeseries.png")
    if (out_dir / "history.csv").exists():
        plot_history(out_dir / "history.csv", out_dir / "loss_curve.png")
    logger.info(f"Metrics and plots written to {out_dir}")
    return results


def main() -> None:
    parser = add_common_args(argparse.ArgumentParser(description="Evaluate a trained ConvLSTM checkpoint."))
    parser.add_argument("--checkpoint", default=None, help="Checkpoint path (default: <output_dir>/best_model.pt)")
    args = parser.parse_args()
    evaluate(load_config(args.config, args.overrides), args.checkpoint)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/convlstm_milano/configs/default.yaml
# Default configuration: full Milano dataset, internet traffic, 1 hour in -> 10 minutes ahead.
# Any value can be overridden from the CLI, e.g. --set training.epochs=20 data.t_out=3
seed: 42
device: auto              # auto | cuda | cpu

paths:                    # relative paths are resolved against the project root
  raw_dir: data/raw
  processed_dir: data/processed
  output_dir: outputs/default

data:
  # Channels stored in the processed tensor (preprocessing). Keep all five so you can switch the
  # input/target channels below without re-running preprocessing.
  features: [smsin, smsout, callin, callout, internet]
  input_features: [internet]       # model input channels, e.g. all five
  target_features: [internet]      # predicted channels (subset of `features`)
  interval_minutes: 10
  grid_size: 100
  max_days: null                   # preprocess only the first N days (null = all)
  fill_method: zero                # zero | ffill_zero  (see src/dataset.py docstring)
  log_transform: false             # apply log1p before scaling (helps with heavy tails)
  scaler: minmax                   # minmax | standard (standard keeps losses near 1 instead of ~1e-6)
  time_features: false             # append sin/cos time-of-day and day-of-week input channels
  mask_inactive: true              # exclude cells with no training-period activity from loss/metrics
  t_in: 6                          # 6 x 10 min = 1 hour of history
  t_out: 1                         # frames to predict
  stride: 1                        # step between training windows
  split: [0.70, 0.15, 0.15]        # chronological train/val/test
  crop: null                       # [row0, row1, col0, col1] to train on a sub-grid
  num_workers: 0                   # DataLoader workers (0 is safest on Windows)

model:
  hidden_dim: 64                   # int, or a list with one entry per layer
  num_layers: 2
  kernel_size: [3, 3]
  bias: true
  refine_layers: 2                 # 3x3 conv refinement in the head (0 = plain 1x1 projection)
  refine_channels: 32
  residual: false                  # predict the change from the last observed frame (tended to get stuck
                                   # at the persistence solution with plain min-max scaling)

training:
  batch_size: 16
  epochs: 100
  lr: 1.0e-3
  weight_decay: 0.0
  loss: mse                        # mse | huber
  huber_delta: 0.05                # on the normalised [0, 1] scale
  patience: 10                     # early stopping on validation loss
  min_delta: 0.0
  min_delta_rel: 0.0               # e.g. 0.001: improvements under 0.1% don't reset patience
  scheduler_patience: 5            # ReduceLROnPlateau
  scheduler_factor: 0.5
  min_lr: 1.0e-6
  grad_clip: 1.0
  amp: true                        # mixed precision (CUDA only)
  max_hours: null                  # stop training before this wall-clock budget is exceeded
  max_train_batches: null          # cap batches per epoch (debugging)
  max_eval_batches: null

evaluation:
  mape_threshold: 1.0              # MAPE only counts cells whose true value exceeds this
  num_plot_samples: 3
  plot_cell: null                  # [row, col] within the (cropped) grid; null = random busy cell
  plot_steps: 432                  # last N test steps in cell_timeseries.png (432 = 3 days); null = all

In [ ]:
%%writefile /kaggle/working/convlstm_milano/configs/kaggle.yaml
# Full run on a Kaggle GPU with the v2 improvements switched on.
_base_: default.yaml

paths:
  raw_dir: /kaggle/input            # replaced by the detected folder (see the notebook)
  processed_dir: /kaggle/temp/processed
  output_dir: /kaggle/working/outputs/convlstm_internet_v2

data:
  num_workers: 2
  scaler: standard                  # mean/std scaling instead of min-max (losses near 1, not 1e-6)
  time_features: true               # sin/cos time-of-day and day-of-week input channels
  log_transform: true               # log1p before scaling: errors become relative, so busy November and
                                    # quieter December are on one footing (persistence train/val loss
                                    # ratio 1.53 -> 1.09 on real data)

training:
  batch_size: 16
  min_delta_rel: 0.001              # improvements under 0.1% don't reset early-stopping patience
  max_hours: 8.5                    # Kaggle sessions are limited to 12 h; leave time for evaluation

## 2. Locate the raw data

In [ ]:
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

# Raw daily files: sms-call-internet-mi-YYYY-MM-DD.(txt|csv)[.gz] anywhere under /kaggle/input
raw_files = sorted(f for f in glob.glob("/kaggle/input/**/sms-call-internet-mi-*", recursive=True)
                   if f.lower().endswith((".txt", ".csv", ".gz")))
assert raw_files, "Attach the dataset dkgmgo/telecom-italia-milan (Add Input -> Datasets)."
raw_dirs = sorted({os.path.dirname(f) for f in raw_files})
assert len(raw_dirs) == 1, f"Raw files are spread over several folders: {raw_dirs}"
RAW_DIR = raw_dirs[0]
print(f"{len(raw_files)} raw files in {RAW_DIR}")
print("first:", os.path.basename(raw_files[0]), "| last:", os.path.basename(raw_files[-1]))

In [ ]:
# Detect the time resolution (the original release is 10-minute; some mirrors are hourly).
import numpy as np, pandas as pd
from src.dataset import read_raw_file
first = read_raw_file(raw_files[0])
gap = pd.Series(np.sort(first["time"].unique())).diff().dropna().min()
INTERVAL = int(gap / pd.Timedelta(minutes=1))
print(f"{len(first):,} (time, cell) records in the first file; time step = {INTERVAL} min; "
      f"{first['cell_id'].nunique():,} distinct cells")
display(first.head())
del first

# Shared overrides for every step, so preprocessing, training and evaluation agree.
SET = f"paths.raw_dir={RAW_DIR} data.interval_minutes={INTERVAL}"
os.environ["PYTHONWARNINGS"] = "ignore"

def run(cmd):
    # Run a command, stream its output into the notebook, and fail loudly on error.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

## 3. Preprocess

Rows are summed over country codes, timestamps converted to Milan local time, and each `CellID`
mapped to `(row, col)` on the 100x100 grid. All five features are stored; missing records mean no
activity, so they are zero-filled. Takes about 11 minutes.

In [ ]:
if os.path.exists("/kaggle/temp/processed/tensor.npy") and os.path.exists("/kaggle/temp/processed/meta.json"):
    print("Processed tensor already exists; skipping preprocessing (delete /kaggle/temp/processed to redo it).")
else:
    run(f"python -m src.dataset --config configs/kaggle.yaml --set {SET}")
meta = json.load(open("/kaggle/temp/processed/meta.json"))
print("tensor shape (T, C, H, W):", meta["shape"], "| features:", meta["features"])
print("days:", meta["days"][0], "->", meta["days"][-1], f"({len(meta['days'])} days)")

## 4. Train

Chronological 70/15/15 split; scaling fitted on the training period only; 6 frames (1 hour) in,
1 frame (10 minutes) out; 2 ConvLSTM layers (hidden 64, 3x3) plus a convolutional head. Each epoch
logs the val loss as a multiple of the persistence loss (below 1.0 means better than "no change").
If the cell is interrupted, re-running it resumes from the last finished epoch.

In [ ]:
run(f"python -m src.train --config configs/kaggle.yaml --resume --set {SET}")

## 5. Evaluate on the test period

In [ ]:
run(f"python -m src.evaluate --config configs/kaggle.yaml --set {SET}")

In [ ]:
OUT = "/kaggle/working/outputs/convlstm_internet_v2"
metrics = json.load(open(f"{OUT}/metrics.json"))
table = pd.DataFrame({name: res["overall"] for name, res in metrics.items() if name != "notes"}).T
table.index.name = "model"
display(table.round(4))
print("ConvLSTM skill vs each baseline (% of the baseline's error removed; positive = better):")
display(pd.DataFrame(metrics["ConvLSTM"]["skill_vs_baselines_%"]).T)
print(json.dumps(metrics["notes"], indent=2))

In [ ]:
# Comparison with run 1 (min-max scaling, no log transform, no calendar inputs), same test period and metrics.
run1 = {"RMSE": 8.4745, "MAE": 3.9071, "MAPE_%": 11.99}
v2 = metrics["ConvLSTM"]["overall"]
cmp = pd.DataFrame({"run 1": run1, "v2": {k: round(v2[k], 4) for k in run1}})
cmp["change %"] = (100 * (cmp["v2"] / cmp["run 1"] - 1)).round(1)
display(cmp)

In [ ]:
from IPython.display import Image
for name in ("loss_curve.png", "heatmaps.png", "cell_timeseries.png"):
    print(name)
    display(Image(filename=f"{OUT}/{name}"))

## Outputs

Everything in `/kaggle/working/outputs/convlstm_internet_v2` is saved with the notebook version:
`best_model.pt`, `metrics.json`, `metrics_per_horizon.csv`, `history.csv`, the logs and the plots.
(`last_checkpoint.pt` is only needed for resuming and can be ignored.)